In [1]:
import torch

print("Is MPS (Apple Silicon GPU) available?", torch.backends.mps.is_available())
print("Is MPS built into this PyTorch binary?", torch.backends.mps.is_built())

Is MPS (Apple Silicon GPU) available? True
Is MPS built into this PyTorch binary? True


### A.2.1 scalar, vector, matrix, tensor

In [ ]:
import torch
import numpy as np

# tensors with python list
tensor0d = torch.tensor(1)
tensor1d = torch.tensor([1,2,3])
tensor2d = torch.tensor([[1,2],
                         [3,4]])
tensor3d = torch.tensor([[[1,2], [3,4]],
                         [[5,6], [7,8]]])

# tensors with numpy
ary3d = np.array([[[1,2], [3,4]],
                 [[5,6], [7,8]]])
tensor3d_2 = torch.tensor(ary3d) # copy numpy array
tensor3d_3 = torch.from_numpy(ary3d) # shares memory with numpy->will change if update

In [4]:
ary3d[0,0,0] = 999
print(tensor3d_2)
print(tensor3d_3)

tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])
tensor([[[999,   2],
         [  3,   4]],

        [[  5,   6],
         [  7,   8]]])


### A.2.2 tensor data types

In [ ]:
tensor1d = torch.tensor([1,2,3])
print(tensor1d.dtype)

floatvec = torch.tensor([1.0,2.0,3.0])
print(floatvec.dtype)

torch.int64
torch.float32


In [7]:
floatvec = tensor1d.to(torch.float32)
print(floatvec.dtype)

torch.float32


### A.2.3 pytorch tensor operations

In [ ]:
tensor2d = torch.tensor([[1,2,3], [4,5,6]])
tensor2d.shape

torch.Size([2, 3])

In [9]:
tensor2d.reshape(3,2)

tensor([[1, 2],
        [3, 4],
        [5, 6]])

In [11]:
tensor2d.view(3,2)

tensor([[1, 2],
        [3, 4],
        [5, 6]])

In [12]:
tensor2d.T

tensor([[1, 4],
        [2, 5],
        [3, 6]])

In [13]:
tensor2d.matmul(tensor2d.T)

tensor([[14, 32],
        [32, 77]])

In [ ]:
tensor2d @ tensor2d.T # also matrix multiplication

tensor([[14, 32],
        [32, 77]])

### A.3 model computation

In [16]:
import torch.nn.functional as F
y = torch.tensor([1.0]) # true label
x1 = torch.tensor([1.1]) # input
w1 = torch.tensor([2.2]) # weight
b = torch.tensor([0.0]) # bias

z = x1 * w1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a,y)
print(loss)

tensor(0.0852)


### A.3 automatic differentiation

In [17]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad = True)
b = torch.tensor([0.0], requires_grad = True)

z = x1 * w1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a,y)

grad_L_w1 = grad(loss, w1, retain_graph = True)
grad_L_b = grad(loss, b, retain_graph = True)

print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [18]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


### A.5 multilayer neural network

In [19]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        self.layers = torch.nn.Sequential(
            # 1st layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),
            # 2nd layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),
            # 3rd layer
            torch.nn.Linear(20, num_outputs),
        )
    def forward(self, x):
        logits = self.layers(x)
        return logits

In [21]:
model = NeuralNetwork(50, 3)
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [23]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Totla number of trainable model parameters:", num_params)
print(model.layers[0].weight)

Totla number of trainable model parameters: 2213
Parameter containing:
tensor([[ 0.0539, -0.0833,  0.1359,  ...,  0.0638,  0.0867,  0.0875],
        [-0.0114, -0.0188, -0.1058,  ..., -0.0097, -0.0852, -0.0593],
        [ 0.0790,  0.0150, -0.0439,  ..., -0.0913, -0.0276, -0.0340],
        ...,
        [-0.1158, -0.0304, -0.0336,  ...,  0.0053,  0.0592,  0.0925],
        [-0.0670,  0.0047,  0.0128,  ..., -0.0223, -0.0633, -0.0914],
        [ 0.0295,  0.0240,  0.0672,  ..., -0.1221,  0.1375, -0.0084]],
       requires_grad=True)


In [24]:
print(model.layers[0].weight.shape)

torch.Size([30, 50])


In [25]:
torch.manual_seed(123)
X = torch.rand((1,50))
out = model(X)
print(out)

tensor([[ 0.1389, -0.1161, -0.2759]], grad_fn=<AddmmBackward0>)


In [26]:
with torch.no_grad():
    out = model(X)
print(out)

tensor([[ 0.1389, -0.1161, -0.2759]])


In [27]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
print(out)

tensor([[0.4106, 0.3182, 0.2712]])


### A.6 efficient data loaders setup

In [ ]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])
y_train = torch.tensor([0,0,0,1,1]) # corresponding class labels

X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])

y_test = torch.tensor([0,1])

In [33]:
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y
    
    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y
    
    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

len(train_ds)

5

In [34]:
from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
    dataset = train_ds,
    batch_size = 2,
    shuffle = True,
    num_workers = 0
)

In [35]:
test_ds = ToyDataset(X_test, y_test)
test_loader = DataLoader(
    dataset = test_ds,
    batch_size = 2,
    shuffle = False,
    num_workers = 0
) # without shuffling

In [36]:
for idx, (x,y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


In [ ]:
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True # ignore the last batch -- convergence disturbance removed
)
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 2: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])


### A.7 typical training loop

In [38]:
import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(num_inputs = 2, num_outputs = 2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(f"Epoch: {epoch+1:03d} / {num_epochs:03d}"
              f" | Batch {batch_idx+1:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")
    model.eval()

Epoch: 001 / 003 | Batch 001/002 | Train/Val Loss: 0.75
Epoch: 001 / 003 | Batch 002/002 | Train/Val Loss: 0.65
Epoch: 002 / 003 | Batch 001/002 | Train/Val Loss: 0.44
Epoch: 002 / 003 | Batch 002/002 | Train/Val Loss: 0.13
Epoch: 003 / 003 | Batch 001/002 | Train/Val Loss: 0.03
Epoch: 003 / 003 | Batch 002/002 | Train/Val Loss: 0.00


In [39]:
model.eval()
with torch.no_grad():
    outputs = model(X_train)
print(outputs)

tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])


In [40]:
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas)

predictions = torch.argmax(probas, dim=1)
print(predictions)

tensor([[0.9991, 0.0009],
        [0.9982, 0.0018],
        [0.9949, 0.0051],
        [0.0491, 0.9509],
        [0.0307, 0.9693]])
tensor([0, 0, 0, 1, 1])


In [41]:
predictions == y_train

tensor([True, True, True, True, True])

In [42]:
torch.sum(predictions == y_train)

tensor(5)

In [43]:
torch.sum(predictions == y_train)

tensor(5)

In [44]:
def compute_accuracy(model, dataloader):
    model.eval()
    correct = 0.0
    total_examples = 0
    for idx, (features, labels) in enumerate(dataloader):
        with torch.no_grad():
            logits = model(features)
        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions
        correct += torch.sum(compare)
        total_examples += len(compare)
    return (correct / total_examples).item()

In [45]:
compute_accuracy(model, train_loader)

1.0

In [46]:
compute_accuracy(model, test_loader)

1.0

### A.8 saving and loading models

In [47]:
torch.save(model.state_dict(), "model.pth")

In [ ]:
model = NeuralNetwork(2,2) #matching original model
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>